# 01 — Data Preparation

**ML-Assisted Electrochemical Descriptor Analysis of Plasmonic Ag/Au g-C3N4 Photocatalysts**

Builds the master table used by every later notebook, by merging the three
source tables (EIS, LSV/CA/Mott-Schottky, and characterization data) and
converting the EIS voltage axis to the RHE reference scale used everywhere
else in this project.

**Inputs:** `Results/TABLE1_EIS_n84.csv`, `Results/TABLE2_catalyst_n28.csv`, `Results/TABLE3_characterization_n7.csv`
**Output:** `Results/MASTER_TABLE_n84.csv` (84 rows x 48 columns)
**Run first** — every other notebook reads `MASTER_TABLE_n84.csv` from disk.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded")

Libraries loaded


## 1. Load the Source Tables

- `TABLE1` — EIS features, one row per catalyst x illumination condition x voltage (84 rows)
- `TABLE2` — LSV, chronoamperometry, and Mott-Schottky features, one row per catalyst x condition (28 rows)
- `TABLE3` — characterization features (TEM, XRD, SSPL, TRPL, UV-Vis), one row per catalyst (7 rows)

In [2]:
df1 = pd.read_csv("Results/TABLE1_EIS_n84.csv")
df2 = pd.read_csv("Results/TABLE2_catalyst_n28.csv")
df3 = pd.read_csv("Results/TABLE3_characterization_n7.csv")

print(f"TABLE1 (EIS):              {df1.shape[0]} rows x {df1.shape[1]} columns")
print(f"TABLE2 (LSV+CA+MS):        {df2.shape[0]} rows x {df2.shape[1]} columns")
print(f"TABLE3 (Characterization): {df3.shape[0]} rows x {df3.shape[1]} columns")
print(f"\nConditions: {df1['condition'].unique()}")
print(f"Voltages (raw, Ag/AgCl):   {sorted(df1['voltage'].unique())}")
print(f"Samples:    {df1['sample'].unique()}")

TABLE1 (EIS):              84 rows x 25 columns
TABLE2 (LSV+CA+MS):        28 rows x 42 columns
TABLE3 (Characterization): 7 rows x 19 columns

Conditions: ['410 nm' '440 nm' 'Dark' 'Full spectrum']
Voltages (raw, Ag/AgCl):   [-1.6, -1.4, -1.2]
Samples:    ['Ag0.25' 'Ag0.5' 'Ag1' 'Au0.25' 'Au0.5' 'Au1' 'C3N4']


## 2. Convert EIS Voltage to the RHE Scale

`TABLE1`'s `voltage` column is recorded on the raw Ag/AgCl scale used by the
potentiostat (-1.2, -1.4, -1.6 V). It is converted once, here, to the RHE
scale used throughout the rest of this project, via

$$E_{RHE} = E_{Ag/AgCl} + 0.971\ \text{V} \quad (\text{pH } 13.1)$$

so every downstream notebook reads a consistent voltage axis (-0.2, -0.4, -0.6 V) directly from the master table.

In [3]:
print(f"Before conversion (Ag/AgCl): {sorted(df1['voltage'].unique())}")

df1["voltage"] = (df1["voltage"] + 0.971).round(1)

print(f"After conversion (RHE):     {sorted(df1['voltage'].unique())}")

Before conversion (Ag/AgCl): [-1.6, -1.4, -1.2]
After conversion (RHE):     [-0.6, -0.4, -0.2]


## 3. Merge into the Master Table (n=84)

`TABLE2` (condition-level) and `TABLE3` (catalyst-level) features are joined
onto `TABLE1` (voltage-level), so every row carries the full descriptor set
for its catalyst x condition x voltage combination.

Each catalyst x condition combination contributes 3 rows here (one per
voltage). For targets that do not vary with voltage (e.g. photocurrent,
Tafel slope), those 3 rows are repeated measurements of the same underlying
quantity rather than independent observations — this is why every model in
this project validates using grouped cross-validation, holding out a full
catalyst x condition group at once, rather than individual rows (see
`utils.py` and Notebook 05).

In [4]:
df2_cols = [
    "sample", "condition", "E_onset_V", "|tafel|_mV_dec",
    "J_m0p6", "J_m0p7", "ΔJ_m0p6", "%enh_m0p6", "ΔJ_m0p7",
    "%enh_m0p7", "CA_J_dark_mA", "CA_J_photo_mA", "CA_dJ_mean_mA",
    "CA_dJ_std_mA", "CA_spike_mA", "CA_stability",
]
df_master = df1.merge(df2[df2_cols], on=["sample", "condition"], how="left")

df3_cols = [
    "sample", "TEM_mean_nm", "TEM_std_nm",
    "XRD_gCN_tau_nm", "XRD_gCN_layers", "XRD_metal_tau_nm",
    "SSPL_quenching_ratio", "TRPL_tau_A_ns", "TRPL_tau_int_ns", "UV_Eg_eV",
]
df_master = df_master.merge(df3[df3_cols], on="sample", how="left")

print(f"Master table shape: {df_master.shape}")
print("\nMissing values by column:")
nan = df_master.isnull().sum()
print(nan[nan > 0].to_string())

Master table shape: (84, 48)

Missing values by column:
Rct_dark            21
delta_Rct           21
Rct_ratio           21
delta_Rs            21
dDeltaRct_dV        21
Rct_ratio_mean      21
delta_Rct_mean      21
ΔJ_m0p6             21
%enh_m0p6           21
ΔJ_m0p7             21
%enh_m0p7           21
CA_J_dark_mA        21
CA_J_photo_mA       21
CA_dJ_mean_mA       27
CA_dJ_std_mA        27
CA_spike_mA         27
CA_stability        27
TEM_mean_nm         12
TEM_std_nm          12
XRD_metal_tau_nm    12


## 4. Fill Structurally-Zero Values

The remaining missing values are all cases where a zero is physically
meaningful rather than genuinely unknown:

- **Dark condition** — `delta_Rct`, `Rct_ratio`, and related photo-response descriptors are zero because there is no illumination and therefore no LSPR response.
- **Bare g-C3N4 (`C3N4`)** — TEM and metal-XRD descriptors are zero because no metal particles are deposited.

No other missing values are imputed.

In [5]:
zero_fill = [
    "delta_Rct", "Rct_ratio", "delta_Rs", "dDeltaRct_dV",
    "Rct_ratio_mean", "delta_Rct_mean", "Rct_dark",
    "ΔJ_m0p6", "%enh_m0p6", "ΔJ_m0p7", "%enh_m0p7",
    "CA_J_dark_mA", "CA_J_photo_mA", "CA_dJ_mean_mA",
    "CA_dJ_std_mA", "CA_spike_mA", "CA_stability",
]
df_master[zero_fill] = df_master[zero_fill].fillna(0)

df_master["TEM_mean_nm"] = df_master["TEM_mean_nm"].fillna(0)
df_master["TEM_std_nm"]  = df_master["TEM_std_nm"].fillna(0)
df_master["XRD_metal_tau_nm"] = df_master["XRD_metal_tau_nm"].fillna(0)

print(f"Missing values remaining: {df_master.isnull().sum().sum()}")
print(f"Final shape: {df_master.shape}")

Missing values remaining: 0
Final shape: (84, 48)


## 5. Save the Master Table

In [6]:
df_master.to_csv("Results/MASTER_TABLE_n84.csv", index=False)
print("Saved: Results/MASTER_TABLE_n84.csv")
print(f"Shape: {df_master.shape[0]} rows x {df_master.shape[1]} columns")

Saved: Results/MASTER_TABLE_n84.csv
Shape: 84 rows x 48 columns


## 6. Preview

The three tables below reproduce the original EIS (n=84), LSV/CA/MS (n=28),
and characterization (n=7) views for a quick visual check of the merge.

In [7]:
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.3f}".format)

eis_cols = [
    "sample", "metal", "loading", "condition", "voltage",
    "Rs_ohm", "Rct_ohm", "peak_freq_Hz", "tau_s", "phase_min",
    "Cdl_est_F", "Rct_dark", "delta_Rct", "Rct_ratio", "delta_Rs",
    "dDeltaRct_dV", "Rct_ratio_mean", "delta_Rct_mean",
    "Vfb_V", "loading_num", "metal_num", "condition_num",
]
print(f"TABLE 1 -- EIS (n=84 rows x {len(eis_cols)} columns)")
display(df_master[eis_cols])

lsv_cols = [
    "sample", "metal", "loading", "condition",
    "Vfb_V", "Eg_eV", "E_onset_V", "|tafel|_mV_dec",
    "J_m0p6", "J_m0p7", "CA_dJ_mean_mA", "CA_dJ_std_mA",
    "CA_spike_mA", "CA_stability", "loading_num", "metal_num", "condition_num",
]
df_lsv = (
    df_master[lsv_cols]
    .drop_duplicates(subset=["sample", "condition"])
    .sort_values(["metal", "loading", "condition"])
    .reset_index(drop=True)
)
print(f"\nTABLE 2 -- LSV + CA + MS (n=28 rows x {len(lsv_cols)} columns)")
display(df_lsv)

char_cols = [
    "sample", "metal", "loading",
    "TEM_mean_nm", "TEM_std_nm", "XRD_gCN_tau_nm",
    "XRD_gCN_layers", "XRD_metal_tau_nm", "SSPL_quenching_ratio",
    "TRPL_tau_A_ns", "TRPL_tau_int_ns", "UV_Eg_eV", "Vfb_V", "Eg_eV",
]
df_char = (
    df_master[char_cols]
    .drop_duplicates(subset=["sample"])
    .sort_values(["metal", "loading"])
    .reset_index(drop=True)
)
print(f"\nTABLE 3 -- Characterization (n=7 rows x {len(char_cols)} columns)")
display(df_char)

TABLE 1 -- EIS (n=84 rows x 22 columns)


,sample,metal,loading,condition,voltage,Rs_ohm,Rct_ohm,peak_freq_Hz,tau_s,phase_min,Cdl_est_F,Rct_dark,delta_Rct,Rct_ratio,delta_Rs,dDeltaRct_dV,Rct_ratio_mean,delta_Rct_mean,Vfb_V,loading_num,metal_num,condition_num
0,Ag0.25,Ag,0.25,410 nm,-0.600,272.231,943.031,19.947,0.008,-39.033,0.000,1286.314,343.284,0.733,-167.381,11290.298,0.589,3085.636,-0.432,0.250,0,1
1,Ag0.25,Ag,0.25,410 nm,-0.400,311.109,3290.476,7.972,0.020,-48.145,0.000,7344.699,4054.222,0.448,16.466,11290.298,0.589,3085.636,-0.432,0.250,0,1
2,Ag0.25,Ag,0.25,410 nm,-0.200,353.608,6861.401,4.971,0.032,-63.547,0.000,11720.803,4859.403,0.585,-26.792,11290.298,0.589,3085.636,-0.432,0.250,0,1
3,Ag0.25,Ag,0.25,440 nm,-0.600,288.668,794.406,19.947,0.008,-37.083,0.000,1286.314,491.909,0.618,-183.818,8440.332,0.610,2576.398,-0.432,0.250,0,2
4,Ag0.25,Ag,0.25,440 nm,-0.400,310.790,3975.453,4.971,0.032,-52.732,0.000,7344.699,3369.245,0.541,16.785,8440.332,0.610,2576.398,-0.432,0.250,0,2
5,Ag0.25,Ag,0.25,440 nm,-0.200,394.215,7852.762,3.942,0.040,-65.298,0.000,11720.803,3868.041,0.670,-67.398,8440.332,0.610,2576.398,-0.432,0.250,0,2
6,Ag0.25,Ag,0.25,Dark,-0.600,104.850,1286.314,15.573,0.010,-59.794,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.432,0.250,0,0
7,Ag0.25,Ag,0.25,Dark,-0.400,327.574,7344.699,1.002,0.159,-64.622,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.432,0.250,0,0
8,Ag0.25,Ag,0.25,Dark,-0.200,326.817,11720.803,1.260,0.126,-71.765,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.432,0.250,0,0
9,Ag0.25,Ag,0.25,Full spectrum,-0.600,97.287,984.018,15.573,0.010,-58.417,0.000,1286.314,302.297,0.765,7.562,582.972,0.836,797.616,-0.432,0.250,0,3



TABLE 2 -- LSV + CA + MS (n=28 rows x 17 columns)


,sample,metal,loading,condition,Vfb_V,Eg_eV,E_onset_V,|tafel|_mV_dec,J_m0p6,J_m0p7,CA_dJ_mean_mA,CA_dJ_std_mA,CA_spike_mA,CA_stability,loading_num,metal_num,condition_num
0,Ag0.25,Ag,0.25,410 nm,-0.432,2.880,-0.900,224.400,-0.241,-0.506,0.002,0.000,0.003,1.068,0.250,0,1
1,Ag0.25,Ag,0.25,440 nm,-0.432,2.880,-0.900,319.400,-0.595,-1.280,0.000,0.000,0.000,0.000,0.250,0,2
2,Ag0.25,Ag,0.25,Dark,-0.432,2.880,-0.898,258.400,-0.186,-0.316,0.000,0.000,0.000,0.000,0.250,0,0
3,Ag0.25,Ag,0.25,Full spectrum,-0.432,2.880,-0.898,191.500,-0.215,-0.361,0.001,0.000,0.000,0.767,0.250,0,3
4,Ag0.5,Ag,0.5,410 nm,-0.856,2.880,-0.900,351.700,-0.676,-1.416,0.003,0.000,0.003,0.865,0.500,0,1
5,Ag0.5,Ag,0.5,440 nm,-0.856,2.880,-0.900,288.400,-0.601,-1.183,0.002,0.000,0.003,0.737,0.500,0,2
6,Ag0.5,Ag,0.5,Dark,-0.856,2.880,-0.900,290.500,-0.316,-0.599,0.000,0.000,0.000,0.000,0.500,0,0
7,Ag0.5,Ag,0.5,Full spectrum,-0.856,2.880,-0.898,253.100,-0.481,-0.859,0.001,0.000,0.001,0.668,0.500,0,3
8,Ag1,Ag,1,410 nm,-0.622,2.840,-0.900,429.800,-0.983,-2.123,0.001,0.000,0.001,0.857,1.000,0,1
9,Ag1,Ag,1,440 nm,-0.622,2.840,-0.900,284.500,-0.504,-1.086,0.001,0.000,0.001,0.608,1.000,0,2



TABLE 3 -- Characterization (n=7 rows x 14 columns)


,sample,metal,loading,TEM_mean_nm,TEM_std_nm,XRD_gCN_tau_nm,XRD_gCN_layers,XRD_metal_tau_nm,SSPL_quenching_ratio,TRPL_tau_A_ns,TRPL_tau_int_ns,UV_Eg_eV,Vfb_V,Eg_eV
0,Ag0.25,Ag,0.25,9.900,1.100,7.053,20.930,1.955,1.228,3.950,17.840,2.880,-0.432,2.880
1,Ag0.5,Ag,0.5,9.600,1.800,7.061,20.950,2.222,0.883,3.800,16.320,2.880,-0.856,2.880
2,Ag1,Ag,1,9.300,2.200,7.078,21.070,2.302,0.415,3.740,17.150,2.840,-0.622,2.840
3,Au0.25,Au,0.25,2.700,0.800,6.972,20.750,2.836,0.843,3.970,17.240,2.860,-1.060,2.860
4,Au0.5,Au,0.5,3.300,1.100,6.970,20.740,2.289,0.697,3.510,16.470,2.850,-0.931,2.850
5,Au1,Au,1,4.200,0.800,7.341,21.780,2.761,0.575,3.840,16.980,2.850,-0.380,2.850
6,C3N4,C3N4,ref,0.000,0.000,6.895,20.520,0.000,1.000,3.540,15.900,2.860,-0.300,2.860
